# 🧬 Enriquecimiento Estratificado de Colas mediante Imputación Inteligente (KNN)

## 🎯 Objetivo del Experimento
Durante el análisis de errores del *Champion Model* (Tecnocasa + IA), detectamos que el modelo presenta una alta precisión en los valores centrales de la distribución (errores en torno al 8-12%), pero pierde capacidad predictiva en las **colas del mercado** (inmuebles a reformar muy baratos o áticos de ultra-lujo), disparando el MAE en esos segmentos por escasez de datos.

Para solucionar esta "ceguera" en los extremos sin contaminar los valores centrales donde el modelo ya es robusto, aplicamos una técnica de **Aumento de Datos Estratificado (*Stratified Data Augmentation*)** inyectando propiedades reales extraídas del portal Redpiso.

## ⚠️ El Reto Técnico: La Varianza de la Inteligencia Artificial
El dataset de Redpiso no contiene las 25 variables generadas por Visión Artificial y NLP (como `nivel_lujo`, `calidad_bano` o `estilo_principal`). 
* **El problema de la mediana:** Imputar estos valores faltantes utilizando la mediana general de las colas destruiría la varianza del dataset (creando un efecto de "línea plana" donde cientos de pisos tendrían la misma información exacta), confundiendo a los algoritmos basados en árboles como XGBoost.
* **La Solución (KNN Imputer):** Para mantener la varianza y la correlación natural, utilizamos el algoritmo de **K-Vecinos Más Cercanos (KNN)**.

## ⚙️ Metodología del Script
1. **Definición de Colas:** Filtramos el dataset de Redpiso para quedarnos exclusivamente con los inmuebles situados en el percentil bajo (< 162.690 €) y el percentil alto (> 416.400 €).
2. **Codificación Categórica:** Transformamos las variables de texto en valores numéricos mediante `OrdinalEncoder` para que el algoritmo matemático pueda procesarlas, guardando los diccionarios para la decodificación posterior.
3. **Búsqueda de "Gemelos Matemáticos":** El `KNNImputer` (con *k=5* y ponderación por distancia) busca los 5 pisos de Tecnocasa más parecidos a cada piso de Redpiso basándose en variables compartidas (Precio, Superficie, Habitaciones, Edad, etc.).
4. **Herencia de IA:** El piso de Redpiso "hereda" las características de IA de sus gemelos, asegurando que un piso barato adquiera puntuaciones de lujo bajas y uno caro puntuaciones premium.
5. **Auditoría Final:** Se realiza un *Sanity Check* para certificar que el proceso ha imputado los datos correctamente, comprobando la coherencia analítica entre la cola baja y la cola alta.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder

print("🧬 INICIANDO ENRIQUECIMIENTO ESTRATIFICADO POR K-VECINOS (KNN)...")

# 1. Cargar datasets
df_tecno = pd.read_parquet("../DATOS/tecnocasa_modelo.parquet", engine='fastparquet')
df_redpiso = pd.read_parquet("../DATOS/redpiso_modelo.parquet", engine='fastparquet')

# 2. Definir los umbrales de las colas
q_bajo = 162690.0
q_alto = 416400.0

redpiso_colas = df_redpiso[(df_redpiso['Precio'] <= q_bajo) | (df_redpiso['Precio'] >= q_alto)].copy()
print(f"Pisos de Redpiso inyectados en las colas: {len(redpiso_colas)}")

# 3. Identificar variables exclusivas de Tecnocasa
cols_ia_faltantes = [col for col in df_tecno.columns if col not in redpiso_colas.columns]
for col in cols_ia_faltantes:
    redpiso_colas[col] = np.nan

# 4. Unificar temporalmente todo para el KNN
df_combinado = pd.concat([df_tecno, redpiso_colas[df_tecno.columns]], ignore_index=True)

# 5. Preparar datos para el KNN
print("⚙️ Preparando el motor KNN (Codificando categóricas)...")
categoricas = df_combinado.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

df_combinado_encoded = df_combinado.copy()
diccionario_encoders = {} # Creamos un diccionario para guardar un codificador por columna

for col in categoricas:
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    mask_not_nan = df_combinado_encoded[col].notna()
    
    # Entrenamos y transformamos, y lo guardamos en el diccionario
    df_combinado_encoded.loc[mask_not_nan, col] = enc.fit_transform(df_combinado.loc[mask_not_nan, [col]]).flatten()
    diccionario_encoders[col] = enc

# 6. APLICAR KNN IMPUTER (SIN DATA LEAKAGE)
print("🧠 Ejecutando KNN Imputer (Buscando gemelos matemáticos SIN mirar el precio)...")

# Identificamos las columnas objetivo que el KNN NO debe ver
columnas_target = ['Precio', 'Precio_m2_Real', 'Log_Precio', 'Precio_m2']
cols_a_ocultar = [c for c in columnas_target if c in df_combinado_encoded.columns]

# Guardamos los precios a salvo temporalmente
df_precios_guardados = df_combinado_encoded[cols_a_ocultar].copy()

# Quitamos los precios del dataset que va a procesar el KNN
df_para_knn = df_combinado_encoded.drop(columns=cols_a_ocultar)

# Ejecutamos el imputador sobre el dataset "ciego" al precio
imputer = KNNImputer(n_neighbors=5, weights='distance')
df_imputado_array = imputer.fit_transform(df_para_knn)

# Reconstruimos el DataFrame imputado
df_imputado = pd.DataFrame(df_imputado_array, columns=df_para_knn.columns)

# Le VOLVEMOS A PEGAR los precios reales que guardamos antes
for col in cols_a_ocultar:
    df_imputado[col] = df_precios_guardados[col].values

# ==============================================================================

# 7. Revertir las categóricas a su texto original
print("🔄 Revirtiendo codificación categórica...")
for col in categoricas:
    enc = diccionario_encoders[col]
    df_imputado[col] = np.round(df_imputado[col])
    categorias_originales = enc.categories_[0]
    
    def revertir_etiqueta(x):
        try:
            x_int = int(x)
            if 0 <= x_int < len(categorias_originales):
                return categorias_originales[x_int]
            return "Desconocido"
        except:
            return "Desconocido"
            
    df_imputado[col] = df_imputado[col].apply(revertir_etiqueta)

# 8. Guardar el Súper-Dataset Final
# Reordenamos las columnas para que queden igual que en df_tecno
df_imputado = df_imputado[df_tecno.columns]

df_imputado.to_parquet("../DATOS/tecnocasa_colas_knn.parquet", engine='fastparquet')
print(f"✅ ¡Éxito! Dataset final (SIN DATA LEAKAGE) creado con shape: {df_imputado.shape}")
print("💾 Guardado como 'tecnocasa_colas_knn.parquet'.")

🧬 INICIANDO ENRIQUECIMIENTO ESTRATIFICADO POR K-VECINOS (KNN)...
Pisos de Redpiso inyectados en las colas: 532
⚙️ Preparando el motor KNN (Codificando categóricas)...
🧠 Ejecutando KNN Imputer (Buscando gemelos matemáticos SIN mirar el precio)...
🔄 Revirtiendo codificación categórica...
✅ ¡Éxito! Dataset final (SIN DATA LEAKAGE) creado con shape: (1387, 66)
💾 Guardado como 'tecnocasa_colas_knn.parquet'.


In [ ]:
import pandas as pd

print("🔍 AUDITORÍA DEL DATASET IMPUTADO (TECNOCASA + COLAS REDPISO)...\n")

# 1. Cargar el dataset final
df_imputado = pd.read_parquet("../DATOS/tecnocasa_colas_knn.parquet", engine='fastparquet')

# 2. Comprobación general
print("=== 1. COMPROBACIÓN GENERAL ===")
print(f"Shape del dataset (Filas, Columnas): {df_imputado.shape}")
nulos_totales = df_imputado.isna().sum().sum()
print(f"Total de valores nulos ocultos: {nulos_totales}")
if nulos_totales > 0:
    print("🚨 Cuidado, han quedado nulos en estas columnas:")
    print(df_imputado.isna().sum()[df_imputado.isna().sum() > 0])
print("-" * 50)

# 3. Revisión de los pisos inyectados (Los últimos de la tabla)
print("\n=== 2. MUESTRA DE PISOS IMPUTADOS (Últimos 5 de Redpiso) ===")
# Elegimos algunas columnas clave que eran exclusivas de la IA de Tecnocasa
# (Si alguna se llamaba distinto, Pandas la ignorará amablemente)
cols_ia_ejemplo = [
    'Precio', 'Superficie', 'Dormitorios', 
    'nivel_lujo', 'nivel_foto_profesional', 'Adjetivos_positivos', 
    'calidad_bano', 'estilo_principal', 'color_principal'
]
cols_mostrar = [c for c in cols_ia_ejemplo if c in df_imputado.columns]

# Mostramos los últimos 5 (que sabemos 100% que son de Redpiso imputados)
display(df_imputado[cols_mostrar].tail(5))
print("-" * 50)

# 4. Prueba de Cordura (Sanity Check) del KNN
# Vamos a comprobar si de verdad el KNN ha sido inteligente separando pobres y ricos
print("\n=== 3. PRUEBA DE INTELIGENCIA DEL KNN ===")
q_bajo = 162690.0
q_alto = 416400.0

cola_baja = df_imputado[df_imputado['Precio'] <= q_bajo]
cola_alta = df_imputado[df_imputado['Precio'] >= q_alto]

if 'nivel_lujo' in df_imputado.columns and 'nivel_foto_profesional' in df_imputado.columns:
    print(f"📉 COLA BAJA (< {q_bajo:,.0f} €):")
    print(f"   - Nivel de Lujo medio: {cola_baja['nivel_lujo'].mean():.2f}")
    print(f"   - Calidad foto media:  {cola_baja['nivel_foto_profesional'].mean():.2f}")
    
    print(f"\n📈 COLA ALTA (> {q_alto:,.0f} €):")
    print(f"   - Nivel de Lujo medio: {cola_alta['nivel_lujo'].mean():.2f}")
    print(f"   - Calidad foto media:  {cola_alta['nivel_foto_profesional'].mean():.2f}")

🔍 AUDITORÍA DEL DATASET IMPUTADO (TECNOCASA + COLAS REDPISO)...

=== 1. COMPROBACIÓN GENERAL ===
Shape del dataset (Filas, Columnas): (1387, 66)
Total de valores nulos ocultos: 0
--------------------------------------------------

=== 2. MUESTRA DE PISOS IMPUTADOS (Últimos 5 de Redpiso) ===


,Precio,Superficie,Dormitorios,nivel_lujo,nivel_foto_profesional,Adjetivos_positivos,calidad_bano,estilo_principal,color_principal
1382,669000.0,70.001577,2.0,3.760030,6.719985,0.599978,estándar,otro,#E0B08B
1383,675972.0,80.798973,3.0,4.159925,6.959911,0.400029,estándar,moderno,#C8553D
1384,680000.0,75.000742,4.0,3.800033,6.560030,0.599967,estándar,moderno,#B22222
1385,695000.0,69.801728,3.0,3.800023,6.759974,0.400023,estándar,moderno,#D3D3D3
1386,725000.0,69.801620,3.0,3.800022,6.759976,0.400022,estándar,moderno,#D3D3D3


--------------------------------------------------

=== 3. PRUEBA DE INTELIGENCIA DEL KNN ===
📉 COLA BAJA (< 162,690 €):
   - Nivel de Lujo medio: 3.72
   - Calidad foto media:  6.55

📈 COLA ALTA (> 416,400 €):
   - Nivel de Lujo medio: 4.00
   - Calidad foto media:  6.78
